In [12]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from captum.attr import LayerAttribution, LayerGradCam, Occlusion
from captum.attr import visualization as viz
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary
from torchvision import transforms


In [13]:
class SimpleCNN(nn.Module):
    """Simple CNN for binary classification of cats vs dogs."""

    def __init__(self, num_classes=2):
        super(SimpleCNN, self).__init__()

        # Convolutional Block 1
        # Input: (batch_size, 3, H, W) - RGB images
        # Output: (batch_size, 32, H/2, W/2) - 32 feature maps, half the spatial size
        self.conv_block1 = nn.Sequential(
            # First conv: 3 input channels (RGB) → 32 output feature maps
            # kernel_size=3: uses 3x3 filters, padding=1: maintains spatial dimensions
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),  # Normalizes the 32 feature maps for stable training
            nn.ReLU(),  # Non-linear activation function
            # Second conv: 32 → 32 feature maps, learns more complex patterns
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # Downsample by 2x: reduces spatial dimensions (e.g., 224x224 → 112x112)
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Convolutional Block 2
        # Input: (batch_size, 32, H/2, W/2)
        # Output: (batch_size, 64, H/4, W/4) - doubles feature maps, halves spatial size
        self.conv_block2 = nn.Sequential(
            # Increase depth: 32 → 64 feature maps for learning more complex features
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # Second conv at this depth level
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # Downsample again: (e.g., 112x112 → 56x56)
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Convolutional Block 3
        # Input: (batch_size, 64, H/4, W/4)
        # Output: (batch_size, 128, 4, 4) - highest-level features, fixed 4x4 spatial size
        self.conv_block3 = nn.Sequential(
            # Increase depth: 64 → 128 feature maps for high-level feature learning
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # Second conv at this depth level
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # Final spatial downsampling (e.g., 56x56 → 28x28)
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Adaptive pooling: converts any spatial size to fixed 4x4
            # This makes the network flexible to different input image sizes
            nn.AdaptiveAvgPool2d((4, 4)),
        )

        # Fully Connected Layers (Classifier)
        # Input: (batch_size, 128, 4, 4) = flattened to (batch_size, 2048)
        # Output: (batch_size, 2) - raw scores for cat and dog classes
        self.classifier = nn.Sequential(
            # Flatten 3D feature maps into 1D vector: 128×4×4 = 2048 features
            nn.Flatten(),
            # First dense layer: 2048 → 512 neurons
            nn.Linear(128 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),  # Randomly drop 50% of neurons during training to prevent overfitting
            # Second dense layer: 512 → 256 neurons
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),  # Another dropout layer for regularization
            # Output layer: 256 → 2 class scores (logits for cat and dog)
            nn.Linear(256, 2),
        )

    def forward(self, x):
        """
        Forward pass through the network.

        Args:
            x: Input tensor of shape (batch_size, 3, H, W)

        Returns:
            Output tensor of shape (batch_size, 2) containing class logits
        """
        # Pass through convolutional blocks to extract features
        x = self.conv_block1(x)  # Extract low-level features (edges, textures)
        x = self.conv_block2(x)  # Extract mid-level features (shapes, patterns)
        x = self.conv_block3(x)  # Extract high-level features (animal parts, structures)

        # Pass through classifier to get final predictions
        x = self.classifier(x)  # Convert features to class scores

        return x  # Returns raw logits (use with CrossEntropyLoss or apply softmax for probabilities)

In [14]:
class CatsDogsDataset(Dataset):
    """PyTorch Dataset for Cats vs Dogs."""

    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        # Convert numpy array to PIL Image if needed
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image.astype("uint8"))

        if self.transform:
            image = self.transform(image)

        return image, label

In [15]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [16]:
def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [17]:
def get_predictions(model, dataloader, device):
    """Get all predictions and true labels."""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)

In [18]:
def plot_training_history(history):
    """Plot training and validation loss and accuracy."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Plot loss
    ax1.plot(history["train_loss"], label="Train Loss", marker="o", linewidth=2)
    ax1.plot(history["val_loss"], label="Validation Loss", marker="o", linewidth=2)
    ax1.set_xlabel("Epoch", fontsize=12)
    ax1.set_ylabel("Loss", fontsize=12)
    ax1.set_title("Training and Validation Loss", fontsize=14, fontweight="bold")
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # Plot accuracy
    ax2.plot(history["train_acc"], label="Train Accuracy", marker="o", linewidth=2)
    ax2.plot(history["val_acc"], label="Validation Accuracy", marker="o", linewidth=2)
    ax2.set_xlabel("Epoch", fontsize=12)
    ax2.set_ylabel("Accuracy (%)", fontsize=12)
    ax2.set_title("Training and Validation Accuracy", fontsize=14, fontweight="bold")
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("✓ Training history plot saved as 'training_history.png'\n")

In [19]:
def plot_confusion_matrix(y_true, y_pred, classes=["Cat (0)", "Dog (1)"]):
    """Plot confusion matrix using only matplotlib."""
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(8, 6))

    # Create heatmap using imshow
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Count", rotation=270, labelpad=20)

    # Set ticks and labels
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)

    # Rotate the tick labels for better readability
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    # Add text annotations
    thresh = cm.max() / 2.0
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=14,
            )

    ax.set_xlabel("Predicted Label", fontsize=12, fontweight="bold")
    ax.set_ylabel("True Label", fontsize=12, fontweight="bold")
    ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("✓ Confusion matrix saved as 'confusion_matrix.png'\n")

In [20]:
def print_classification_report(y_true, y_pred, classes=["Cat (0)", "Dog (1)"]):
    """Print and save classification report."""
    print("=" * 80)
    print("CLASSIFICATION REPORT ON TEST DATASET")
    print("=" * 80)

    report = classification_report(y_true, y_pred, target_names=classes, digits=4)
    print(report)

    # Save to file
    with open("classification_report.txt", "w") as f:
        f.write("CLASSIFICATION REPORT ON TEST DATASET\n")
        f.write("=" * 80 + "\n")
        f.write(report)

    print("=" * 80)
    print("✓ Classification report saved as 'classification_report.txt'\n")

In [ ]:
def prepare_image(image, device):
    """Helper function to prepare image for visualization."""
    if isinstance(image, Image.Image):
        from torchvision import transforms

        transform = transforms.Compose(
            [
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Using ImageNet standard normalization values
            ]
        )
        input_tensor = transform(image).unsqueeze(0).to(device)
        # Keep transformed but not normalized for visualization
        transformed_img = transforms.Compose(
            [
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
            ]
        )(image)
    else:
        input_tensor = image.to(device)
        # Denormalize for visualization
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        transformed_img = (image[0].cpu() * std + mean).clamp(0, 1)

    return input_tensor, transformed_img


def apply_occlusion_to_batch(images, patch_size, stride=8):
    """
    Apply systematic occlusion to a batch of images.
    
    Args:
        images: Tensor of shape (B, C, H, W)
        patch_size: Size of occlusion patch
        stride: Stride for patch placement
    
    Returns:
        Occluded images tensor
    """
    occluded_images = images.clone()
    B, C, H, W = images.shape
    
    # Apply occlusion patches at regular intervals
    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            # Randomly occlude some patches (30% probability)
            if torch.rand(1).item() < 0.3:
                occluded_images[:, :, y:y+patch_size, x:x+patch_size] = 0
    
    return occluded_images


def evaluate_model_accuracy(model, test_loader, device, patch_size=None, stride=8, class_names=["Cat", "Dog"]):
    """
    Evaluate model accuracy with optional occlusion.
    
    Args:
        model: PyTorch model
        test_loader: Test data loader
        device: torch device
        patch_size: Size of occlusion patch (None for no occlusion)
        stride: Stride for occlusion
        class_names: List of class names
    
    Returns:
        Accuracy percentage
    """
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Apply occlusion if specified
            if patch_size is not None:
                images = apply_occlusion_to_batch(images, patch_size, stride)
            
            # Get predictions
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    return accuracy


def evaluate_occlusion_impact(model, test_loader, device, patch_sizes=[15, 25, 40], stride=8, class_names=["Cat", "Dog"]):
    """
    Evaluate model accuracy with different occlusion patch sizes.
    
    Args:
        model: PyTorch model
        test_loader: Test data loader
        device: torch device
        patch_sizes: List of occlusion patch sizes
        stride: Stride for occlusion
        class_names: List of class names
    
    Returns:
        Dictionary with accuracy results
    """
    print("\n" + "=" * 80)
    print("OCCLUSION IMPACT ON MODEL ACCURACY")
    print("=" * 80)
    
    # Baseline accuracy (no occlusion)
    print("\nEvaluating baseline accuracy (no occlusion)...")
    baseline_acc = evaluate_model_accuracy(model, test_loader, device, patch_size=None)
    print(f"✓ Baseline Accuracy: {baseline_acc:.2f}%")
    
    results = {
        "baseline": baseline_acc,
        "occluded": {}
    }
    
    # Test each occlusion size
    print("\nEvaluating with different occlusion patch sizes:")
    print("-" * 80)
    
    for patch_size in patch_sizes:
        print(f"\nTesting {patch_size}x{patch_size} occlusion patches...")
        acc = evaluate_model_accuracy(model, test_loader, device, patch_size=patch_size, stride=stride)
        drop = baseline_acc - acc
        results["occluded"][patch_size] = acc
        
        print(f"  • Accuracy: {acc:.2f}%")
        print(f"  • Accuracy Drop: {drop:.2f}%")
        print(f"  • Relative Performance: {(acc/baseline_acc)*100:.1f}% of baseline")
    
    # Create visualization
    create_accuracy_comparison_plot(results, patch_sizes)
    
    # Print summary
    print("\n" + "=" * 80)
    print("ACCURACY SUMMARY:")
    print("=" * 80)
    print(f"Baseline (No Occlusion):  {results['baseline']:.2f}%")
    for size in patch_sizes:
        print(f"Occlusion {size}x{size}:         {results['occluded'][size]:.2f}% "
              f"(↓{results['baseline'] - results['occluded'][size]:.2f}%)")
    print("=" * 80 + "\n")
    
    return results


def create_accuracy_comparison_plot(results, patch_sizes):
    """Create bar plot comparing accuracies."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    labels = ['Baseline'] + [f'{size}x{size}' for size in patch_sizes]
    accuracies = [results['baseline']] + [results['occluded'][size] for size in patch_sizes]
    colors = ['#2ecc71'] + ['#e74c3c'] * len(patch_sizes)
    
    bars = ax.bar(labels, accuracies, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar, acc in zip(bars, accuracies):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.1f}%',
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Occlusion Type', fontsize=12, fontweight='bold')
    ax.set_title('Model Accuracy: Baseline vs. Occlusion', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig("occlusion_accuracy_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()


def visualize_gradcam_captum(model, image, device, layers_dict, class_names=["Cat", "Dog"]):
    """
    Generate GradCAM visualizations using Captum.

    Args:
        model: PyTorch model
        image: Input tensor (1, C, H, W) or PIL Image
        device: torch device
        layers_dict: Dict of {layer_name: layer_object}
        class_names: List of class names
    """
    model.eval()

    # Prepare image
    input_tensor, transformed_img = prepare_image(image, device)

    # Get prediction
    with torch.no_grad():
        output = model(input_tensor)
        pred_class = output.argmax(dim=1).item()
        pred_prob = F.softmax(output, dim=1)[0, pred_class].item()

    print(f"\nPredicted Class: {class_names[pred_class]} (Confidence: {pred_prob:.2%})")
    print("\nGradCAM Analysis:")
    print("-" * 80)

    # Create figure with subplots for each layer
    num_layers = len(layers_dict)
    fig, axes = plt.subplots(1, num_layers, figsize=(6 * num_layers, 6))
    if num_layers == 1:
        axes = [axes]

    # Generate GradCAM for each layer
    for idx, (layer_name, layer) in enumerate(layers_dict.items()):
        print(f"\nAnalyzing {layer_name}...")

        # Create LayerGradCam instance
        layer_gc = LayerGradCam(model, layer)

        # Generate attribution
        attributions = layer_gc.attribute(input_tensor, target=pred_class)

        # Upsample to input size
        upsampled_attr = LayerAttribution.interpolate(attributions, (224, 224))

        # Convert to numpy - shape is (1, C, H, W), need (H, W, C) for visualization
        attr_data = upsampled_attr.squeeze(0).cpu().detach().numpy()
        attr_data = np.transpose(attr_data, (1, 2, 0))  # (H, W, C)

        # Convert transformed_img to numpy - shape is (C, H, W), need (H, W, C)
        original_img_data = np.transpose(transformed_img.cpu().numpy(), (1, 2, 0))

        print(f"  • Attribution shape: {attr_data.shape}")

        # Use Captum's visualizer
        _ = viz.visualize_image_attr(
            attr_data,
            original_img_data,
            method="blended_heat_map",
            sign="positive",
            show_colorbar=False,
            title=f"{layer_name}\n{class_names[pred_class]}: {pred_prob:.1%}",
            plt_fig_axis=(fig, axes[idx]),
            use_pyplot=False,
        )

        # Print interpretation
        max_intensity = np.abs(attr_data).max()
        print(f"  • Max activation intensity: {max_intensity:.3f}")

    plt.tight_layout()
    plt.savefig("gradcam_captum_visualization.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Print layer-specific insights
    print("\n" + "=" * 80)
    print("LAYER-BY-LAYER INTERPRETATION:")
    print("=" * 80)
    print("• Conv Block 1 (Early Layer):")
    print("  - Detects LOW-LEVEL features: edges, textures, basic shapes, color patterns")
    print("  - Activates across large areas of the image")
    print("  - Foundation for higher-level feature detection")
    print("\n• Conv Block 2 (Middle Layer):")
    print("  - Detects MID-LEVEL features: object parts, patterns, facial features")
    print("  - Begins to localize important regions (ears, eyes, nose, fur patterns)")
    print("  - More selective than early layers")
    print("\n• Conv Block 3 (Deep Layer):")
    print("  - Detects HIGH-LEVEL semantic features: whole objects, faces, body parts")
    print("  - Highly selective - focuses on most discriminative regions")
    print("  - Directly influences final classification decision")
    print("=" * 80 + "\n")


def visualize_occlusion_captum(model, image, device, patch_sizes=[15, 25, 40], stride=8, class_names=["Cat", "Dog"]):
    """
    Generate Occlusion Sensitivity visualizations using Captum.

    Args:
        model: PyTorch model
        image: Input tensor (1, C, H, W) or PIL Image
        device: torch device
        patch_sizes: List of occlusion patch sizes (square patches)
        stride: Stride for sliding window
        class_names: List of class names
    """
    model.eval()

    # Prepare image
    input_tensor, transformed_img = prepare_image(image, device)

    # Get baseline prediction
    with torch.no_grad():
        output = model(input_tensor)
        pred_class = output.argmax(dim=1).item()
        baseline_prob = F.softmax(output, dim=1)[0, pred_class].item()

    print("\n" + "=" * 80)
    print("OCCLUSION SENSITIVITY ANALYSIS")
    print("=" * 80)
    print(f"Baseline Prediction: {class_names[pred_class]} ({baseline_prob:.2%})")
    print("\nTesting different occlusion patch sizes:")
    print("-" * 80)

    # Create figure with subplots
    fig, axes = plt.subplots(1, len(patch_sizes), figsize=(6 * len(patch_sizes), 6))
    if len(patch_sizes) == 1:
        axes = [axes]

    results = []

    # Convert transformed_img to numpy for visualization
    original_img_data = np.transpose(transformed_img.cpu().numpy(), (1, 2, 0))

    # Test each patch size
    for idx, patch_size in enumerate(patch_sizes):
        print(f"\nPatch Size: {patch_size}x{patch_size} pixels (Stride: {stride})")

        # Create Occlusion instance
        occlusion = Occlusion(model)

        # Generate occlusion attribution
        attributions = occlusion.attribute(
            input_tensor,
            target=pred_class,
            sliding_window_shapes=(3, patch_size, patch_size),
            strides=(3, stride, stride),
            baselines=0,  # Black occlusion patch
        )

        # Convert to numpy - shape is (1, C, H, W), need (H, W, C)
        attr_data = attributions.squeeze(0).cpu().detach().numpy()
        attr_data = np.transpose(attr_data, (1, 2, 0))  # (H, W, C)

        print(f"  • Attribution shape: {attr_data.shape}")

        # Use Captum's visualizer
        _ = viz.visualize_image_attr(
            attr_data,
            original_img_data,
            method="heat_map",
            sign="absolute_value",
            show_colorbar=False,
            title=f"Patch: {patch_size}x{patch_size}\n{class_names[pred_class]}: {baseline_prob:.1%}",
            plt_fig_axis=(fig, axes[idx]),
            use_pyplot=False,
            cmap="hot",
        )

        # Calculate statistics
        attr_magnitude = np.abs(attr_data).mean(axis=2)  # Average across channels
        attr_min, attr_max = attr_magnitude.min(), attr_magnitude.max()
        high_importance = (attr_magnitude > np.percentile(attr_magnitude, 70)).sum() / attr_magnitude.size

        print(f"  • Attribution range: [{attr_min:.4f}, {attr_max:.4f}]")
        print(f"  • High importance regions: {high_importance:.1%} of image")

        results.append(
            {"patch_size": patch_size, "attr_range": (attr_min, attr_max), "high_importance_pct": high_importance}
        )

    plt.tight_layout()
    plt.savefig("occlusion_captum_sensitivity.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Print comprehensive interpretation
    print("\n" + "=" * 80)
    print("INTERPRETATION OF RESULTS:")
    print("=" * 80)
    print("\nWhat the Colors Mean:")
    print("  • RED/BRIGHT areas: Occluding these regions GREATLY reduces model confidence")
    print("  • YELLOW areas: Moderately important for classification")
    print("  • DARK areas: Less important - occluding has minimal effect")

    print("\nPatch Size Comparison:")
    print(f"  • Small patches ({patch_sizes[0]}x{patch_sizes[0]}): Precise localization")
    print(f"  • Medium patches ({patch_sizes[1]}x{patch_sizes[1]}): Balance of precision and context")
    print(f"  • Large patches ({patch_sizes[2]}x{patch_sizes[2]}): Broad region importance")
    
    print("=" * 80 + "\n")

    return results


def analyze_model_interpretability_captum(model, test_loader, device, num_samples=3):
    """
    Complete interpretability analysis using Captum.

    Args:
        model: Trained PyTorch model
        test_loader: Test data loader
        device: torch device
        num_samples: Number of test images to analyze
    """
    model.eval()

    # STEP 1: Evaluate accuracy with different occlusion masks
    print("\n" + "=" * 80)
    print("STEP 1: ACCURACY EVALUATION WITH OCCLUSION")
    print("=" * 80)
    
    accuracy_results = evaluate_occlusion_impact(
        model, test_loader, device, 
        patch_sizes=[15, 25, 40], 
        stride=8
    )

    # STEP 2: Visual analysis on sample images
    print("\n" + "=" * 80)
    print("STEP 2: DETAILED VISUAL ANALYSIS ON SAMPLE IMAGES")
    print("=" * 80)

    # Get sample images
    images, labels = next(iter(test_loader))

    # Define layers to visualize
    layers_dict = {
        "Conv Block 1": model.conv_block1,
        "Conv Block 2": model.conv_block2,
        "Conv Block 3": model.conv_block3,
    }

    class_names = ["Cat", "Dog"]

    for i in range(min(num_samples, len(images))):
        print("\n" + "=" * 80)
        print(f"ANALYZING SAMPLE IMAGE {i + 1}/{num_samples}")
        print("=" * 80)
        print(f"True Label: {class_names[labels[i]]}")

        # GRADCAM ANALYSIS
        print("\n" + "=" * 80)
        print("PART A: GRADCAM VISUALIZATION")
        print("=" * 80)

        visualize_gradcam_captum(model, images[i : i + 1], device, layers_dict, class_names)

        # OCCLUSION SENSITIVITY ANALYSIS
        print("\n" + "=" * 80)
        print("PART B: OCCLUSION SENSITIVITY ANALYSIS")
        print("=" * 80)

        occlusion_results = visualize_occlusion_captum(
            model, images[i : i + 1], device, patch_sizes=[15, 25, 40], stride=8, class_names=class_names
        )

    # STEP 3: Final Summary
    print("\n" + "=" * 80)
    print("FINAL SUMMARY")
    print("=" * 80)
    print("\n✓ Completed GradCAM analysis showing layer-wise feature extraction")
    print("✓ Completed Occlusion Sensitivity analysis with 3 different patch sizes")
    print("✓ Generated heat maps for each occlusion mask size")
    print(f"✓ Evaluated model accuracy across {len(accuracy_results['occluded'])} occlusion conditions")
    print("\nGenerated Files:")
    print("  • gradcam_captum_visualization.png - GradCAM heat maps")
    print("  • occlusion_captum_sensitivity.png - Occlusion heat maps")
    print("  • occlusion_accuracy_comparison.png - Accuracy comparison plot")
    print("=" * 80 + "\n")
    
    return accuracy_results


# Example usage:
# accuracy_results = analyze_model_interpretability_captum(model, test_loader, device, num_samples=3)

In [ ]:
# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
NUM_CLASSES = 2

# Control flags
FORCE_RETRAIN = False  # Set to True to retrain even if model exists
MODEL_PATH = "best_model.pth"

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")


# ============================================================================
# HELPER: Check if we should skip training
# ============================================================================
def should_skip_training():
    """Check if model exists and we shouldn't force retrain."""
    if FORCE_RETRAIN:
        print("⚠️ FORCE_RETRAIN is True - will retrain model")
        return False

    if os.path.exists(MODEL_PATH):
        print(f"Found existing model at '{MODEL_PATH}'")
        print("  Set FORCE_RETRAIN=True to retrain")
        return True

    print("✗ No existing model found - will train")
    return False


# ============================================================================
# 1. LOAD AND PREPARE DATA
# ============================================================================
print("=" * 80)
print("DATA LOADING")
print("=" * 80)
print("Loading dataset")
data = load_dataset("pantelism/cats-vs-dogs")

print("Preparing images...")
train_images = [img for img in data["train"]["image"]]
train_labels = np.array(data["train"]["label"])

print(f"Total images: {len(train_images)}")
print(f"Label distribution: {np.unique(train_labels, return_counts=True)}\n")

# Split into train, validation, and test
train_imgs, temp_imgs, train_lbls, temp_lbls = train_test_split(
    train_images, train_labels, test_size=0.3, random_state=42, stratify=train_labels
)

val_imgs, test_imgs, val_lbls, test_lbls = train_test_split(
    temp_imgs, temp_lbls, test_size=0.5, random_state=42, stratify=temp_lbls
)

print(f"\nTrain set: {len(train_imgs)} images")
print(f"Validation set: {len(val_imgs)} images")
print(f"Test set: {len(test_imgs)} images\n")

# ============================================================================
# 2. CREATE DATASETS AND DATALOADERS
# ============================================================================
print("Creating datasets...")

# Data transforms
train_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# Create datasets and dataloaders
train_dataset = CatsDogsDataset(train_imgs, train_lbls, transform=train_transform)
val_dataset = CatsDogsDataset(val_imgs, val_lbls, transform=val_transform)
test_dataset = CatsDogsDataset(test_imgs, test_lbls, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("✓ Datasets ready!\n")

# ============================================================================
# 3. CREATE MODEL
# ============================================================================
model = SimpleCNN(num_classes=NUM_CLASSES)
model = model.to(device)

# Print model summary, must print summary or else it does not show on notebooks
print(model)
print("\n" + "=" * 80)
print("MODEL SUMMARY")
print("=" * 80)
print(
    summary(
        model,
        input_size=(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE),
        col_names=["input_size", "output_size", "num_params"],
        depth=3,
    )
)
print("=" * 80 + "\n")

# ============================================================================
# 4. TRAIN THE MODEL (OR SKIP IF MODEL EXISTS)
# ============================================================================
if should_skip_training():
    print("\nSKIPPING TRAINING - Loading existing model\n")
    model.load_state_dict(torch.load(MODEL_PATH))
    print(f"Model loaded from '{MODEL_PATH}'\n")

else:
    print("\n" + "=" * 80)
    print("TRAINING MODEL")
    print("=" * 80)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    print("Starting training...\n")
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    best_val_acc = 0.0

    for epoch in range(EPOCHS):
        print(f"Epoch {epoch + 1}/{EPOCHS}")
        print("-" * 40)

        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

        # Validate
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        # Learning rate scheduling
        scheduler.step(val_loss)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), MODEL_PATH)
            print("Model saved!")
        print()

    print(f"Training complete! Best validation accuracy: {best_val_acc:.2f}%\n")

    # Plot training history
    plot_training_history(history)

    # Load best model
    model.load_state_dict(torch.load(MODEL_PATH))

# ============================================================================
# 5. EVALUATE ON TEST SET
# ============================================================================
print("\n" + "=" * 80)
print("TEST SET EVALUATION")
print("=" * 80)

criterion = nn.CrossEntropyLoss()

print("Evaluating on test set...")
test_loss, test_acc = validate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}\n")

# Confusion Matrix
print("Generating predictions...")
y_pred, y_true = get_predictions(model, test_loader, device)
plot_confusion_matrix(y_true, y_pred, classes=["Cat (0)", "Dog (1)"])

# Classification Report
print_classification_report(y_true, y_pred, classes=["Cat (0)", "Dog (1)"])

# ========================================================================
# SUMMARY
# ========================================================================

print(f"\nFinal Test Accuracy: {test_acc:.2f}%")
print("=" * 80)

# TASK 2
print("\\n" + "=" * 80)
print("INTERPRETABILITY ANALYSIS WITH CAPTUM")
print("=" * 80)

model.load_state_dict(torch.load("best_model.pth"))
model.eval()

analyze_model_interpretability_captum(model, test_loader, device, num_samples=3)

print("\\n✓ Interpretability analysis complete!")
print("Saved visualizations:")
print("  - gradcam_captum_visualization.png")
print("  - occlusion_captum_sensitivity.png")

Using device: cuda

DATA LOADING
Loading dataset


Preparing images...
